In [1]:
import pandas as pd
import numpy as np
import itertools
import functools
import joblib
import gc

import pywt
import math

from functools import reduce

from sklearn.linear_model import LinearRegression

from scipy.ndimage import median_filter
from scipy.optimize import minimize
from scipy.ndimage import gaussian_filter1d
from scipy.signal import find_peaks

import matplotlib.pyplot as plt

from tqdm import tqdm

mode = 'test'
# mode = 'train'

In [2]:
def sigma_clip_custom(data, sigma=3, maxiters=5):
    mask = np.zeros_like(data, dtype=bool)
    
    for _ in range(maxiters):
        if np.all(~mask):
            mean = data.mean()
            std = data.std()
        else:
            mean = data[~mask].mean()
            std = data[~mask].std()
        
        new_mask = (data > mean + sigma * std) | (data < mean - sigma * std)
        updated_mask = mask | new_mask
        
        if np.array_equal(updated_mask, mask):
            break
        
        mask = updated_mask
    
    return mask

def apply_linear_corr(linear_corr, clean_signal):
    linear_corr = np.flip(linear_corr, axis=0)
    for x, y in itertools.product(
                range(clean_signal.shape[1]), range(clean_signal.shape[2])
            ):
        poli = np.poly1d(linear_corr[:, x, y])
        clean_signal[:, x, y] = poli(clean_signal[:, x, y])
    return clean_signal

def clean_dark(signal, dark, dt):
    dark = np.tile(dark, (signal.shape[0], 1, 1))
    signal -= dark* dt[:, np.newaxis, np.newaxis]
    return signal

def preproc_airs(planet_id, adc_info, dataset='train'):
    cut_inf, cut_sup = 39, 321
    original_sensor_size = [11250, 32, 356]
#     target_sensor_size = [1, 32, cut_sup-cut_inf]
    target_sensor_size = [1, 32, 356]
    sensor = 'AIRS-CH0'
    linear_corr = (6, 32, 356)

    
    signal = pd.read_parquet(f'/kaggle/input/ariel-data-challenge-2024/{dataset}/{planet_id}/{sensor}_signal.parquet').to_numpy()
    dark_frame = pd.read_parquet(f'/kaggle/input/ariel-data-challenge-2024/{dataset}/' + str(planet_id) + '/' + sensor + '_calibration/dark.parquet', engine='pyarrow').to_numpy()
    dead_frame = pd.read_parquet(f'/kaggle/input/ariel-data-challenge-2024/{dataset}/' + str(planet_id) + '/' + sensor + '_calibration/dead.parquet', engine='pyarrow').to_numpy()
    flat_frame = pd.read_parquet(f'/kaggle/input/ariel-data-challenge-2024/{dataset}/' + str(planet_id) + '/' + sensor + '_calibration/flat.parquet', engine='pyarrow').to_numpy()
    linear_corr = pd.read_parquet(f'/kaggle/input/ariel-data-challenge-2024/{dataset}/' + str(planet_id) + '/' + sensor + '_calibration/linear_corr.parquet').values.astype(np.float64).reshape(linear_corr)

    signal = signal.reshape(original_sensor_size) 
    
    gain = adc_info.loc[planet_id][f'{sensor}_adc_gain']
    offset = adc_info.loc[planet_id][f'{sensor}_adc_offset']
    
    signal = signal / gain + offset

    hot = sigma_clip_custom(dark_frame, sigma=5, maxiters=5)
#     hot = sigma_clip(dark_frame, sigma=5, maxiters=5).mask

    dt = np.ones(len(signal))*0.1
    dt[1::2] += 4.5 

    signal = signal.clip(0)
    linear_corr_signal = apply_linear_corr(linear_corr, signal)
    signal = clean_dark(linear_corr_signal, dark_frame, dt)

    flat = flat_frame.reshape(target_sensor_size)
    flat[dead_frame.reshape(target_sensor_size)] = np.nan
    flat[hot.reshape(target_sensor_size)] = np.nan
    signal = signal / flat
    
    signal = signal[:,10:22,:]

    mean_signal = np.nanmean(signal, axis=1) 
    cds_signal = (mean_signal[1::2] - mean_signal[0::2])

    return cds_signal

In [3]:
wavelenghts_file =  f'/kaggle/input/ariel-data-challenge-2024/wavelengths.csv'
adc_info_file =  f'/kaggle/input/ariel-data-challenge-2024/{mode}_adc_info.csv'

# y_file =  f'/kaggle/input/ariel-data-challenge-2024/train_labels.csv'
# y_df = pd.read_csv(y_file, index_col='planet_id')
wavelenghts_df =  f'/kaggle/input/ariel-data-challenge-2024/wavelengths.csv'

adc_info = pd.read_csv(adc_info_file, index_col='planet_id')
planet_ids = adc_info.index.to_list()

In [4]:
# Utilities functions
def linear_transform(signal):
    x = np.arange(len(signal)).reshape(-1, 1)
    reg = LinearRegression().fit(x, signal)
    trend_line = reg.predict(x).flatten()
    signal_transformed = signal - trend_line    
    return signal_transformed, trend_line

def extend_signal_with_polyfit(x, y, extend_points, deg):
    p = np.polyfit(x, y, deg)
    x_left = np.linspace(x[0] - extend_points, x[0], extend_points)
    y_left = np.polyval(p, x_left)
    x_right = np.linspace(x[-1], x[-1] + extend_points, extend_points)
    y_right = np.polyval(p, x_right)
    x_extended = np.concatenate([x_left, x, x_right])
    y_extended = np.concatenate([y_left, y, y_right])

    return x_extended, y_extended

def polynomial_transform2(signal, degree, K, L):
    x_start = np.arange(K)
    x_end = np.arange(L, len(signal))

    y_start = signal[:K]
    y_end = signal[L:]

    x_concat = np.concatenate([x_start, x_end])
    y_concat = np.concatenate([y_start, y_end])

    extend_points = 300
    x = np.arange(len(signal))
    x, signal_extended = extend_signal_with_polyfit(x_concat, y_concat, extend_points, deg=degree-1)

    coefs = np.polyfit(x, signal_extended, degree)
    
    x_full = np.arange(len(signal))
    trend_poly = np.polyval(coefs, x_full)

    signal_transformed = signal - trend_poly

    return signal_transformed, trend_poly


def refit_with_polyfit(x, y, deg):
    nn = len(x)
    w = np.ones(nn)
    # w[ : nn // 2] = np.linspace(1, 0.9, nn // 2)
    # w[nn // 2 : ] = np.linspace(0.9, 1, len(w[nn // 2:]))
    p = np.polyfit(x, y, deg, w=w)

    return np.polyval(p, x)

def wavelet_denoising(signal, wavelet='db29', strength=1.0, mode='symmetric', level=None):
    coeff = pywt.wavedec(signal, wavelet, mode=mode, level=level)
    coeff[1:] = [pywt.threshold(c, strength * np.std(c)) for c in coeff[1:]]
    return pywt.waverec(coeff, wavelet, mode=mode)

def polyfit_denoising(signal, extend_points=100, max_deg=4):
    x = np.arange(len(signal))

    if len(signal) < 2 * extend_points:
        return signal
    x_extended, signal_extended = extend_signal_with_polyfit(x, signal, extend_points, deg=1)

    for deg in range(2, max_deg+1):
        signal_refit = refit_with_polyfit(x_extended, signal_extended, deg)
        signal_extended[:extend_points] = signal_refit[:extend_points]
        signal_extended[-extend_points:] = signal_refit[-extend_points:]

    signal_refit = refit_with_polyfit(x_extended, signal_extended, max_deg)

    return signal_refit[extend_points:-extend_points]

def transit_formula(s):
    return 1/s - 1

In [5]:
# Transformation tools

def dummy(**kwargs):
    return kwargs


def bin_wavelengths(**kwargs):
    fromwl = kwargs['from_wavelength']
    towl = kwargs['to_wavelength']
    signal = kwargs['signal']
    kwargs['signal'] = signal[:, fromwl:towl].mean(axis=1)
    return kwargs

def normalize_signal(**kwargs):
    # signal = signal.copy()
    # wavelength_means = signal[:, :].mean(axis=0)
    # signal[:, :] /= wavelength_means
    signal = kwargs['signal']
    normalized_signal = signal.copy()
    wavelength_means = signal.mean(axis=0)
    normalized_signal /= wavelength_means
    kwargs['signal'] = normalized_signal
    return kwargs

def shrink_signal(ratio=12, **kwargs):
    signal = kwargs['signal']
    n = signal.shape[0]
    signal_windowed = signal[:n - (n % ratio)].reshape(-1, ratio)
    signal_means = signal_windowed.mean(axis=1)
    signal_stds = signal_windowed.std(axis=1)
    kwargs['signal'] = signal_means
    kwargs['signal_stds'] = signal_stds

    kwargs['margin'] //= ratio
    if 'ingress_left' in kwargs:
        kwargs['ingress_left'] //= ratio
        kwargs['ingress_right'] //= ratio
        kwargs['egress_left'] //= ratio
        kwargs['egress_right'] //= ratio

    return kwargs

def linear_detrend(**kwargs):
    signal = kwargs['signal']
    signal_transformed, trend_line = linear_transform(signal)

    kwargs['signal'] = signal_transformed + trend_line.mean()
    kwargs['trend_line'] = trend_line
    kwargs['signal_before_detrend'] = signal

    return kwargs

def polynomial_detrend(**kwargs):
    signal = kwargs['signal']
    degree = kwargs.get('degree', 5)
    K = kwargs['ingress_left'] - 10
    L = kwargs['egress_right'] + 10

    signal_transformed, trend_poly = polynomial_transform2(signal, degree, K, L)

    kwargs['signal'] = signal_transformed
    kwargs['trend_poly'] = trend_poly

    return kwargs

def get_transit_phases2(**kwargs):
    signal = kwargs['signal'] + 1
    grad1 = np.gradient(signal)
    mn = np.argmin(grad1)
    mx = np.argmax(grad1)
    peaks = find_peaks(grad1, width=30)[0]
    ingress_left = peaks[peaks < mn][-1]
    ingress_right = peaks[peaks > mn][0]
    dips = find_peaks(-grad1, width=20)[0]
    egress_left = dips[dips < mx][-1]
    egress_right = dips[dips > mx][0]

    
    kwargs['ingress_left'] = ingress_left
    kwargs['ingress_right'] = ingress_right
    kwargs['egress_left'] = egress_left
    kwargs['egress_right'] = egress_right

    return kwargs


def get_transit_phases(edge_margin=300, edge_margin2=250, sigma1=100, sigma2=10, **kwargs):
    signal  = kwargs['signal']
    singal_transformed, trend_line = linear_transform(signal)
    signal_denoised = gaussian_filter1d(singal_transformed, sigma=sigma1)
    grad = np.gradient(signal_denoised)
    grad_transformed, _ = linear_transform(grad)
    ingress_mean = np.argmin(grad_transformed[edge_margin:-edge_margin]) + edge_margin
    egress_mean = np.argmax(grad_transformed[edge_margin:-edge_margin]) + edge_margin

    grad2 = np.gradient(grad_transformed)
    grad2_denoised = gaussian_filter1d(grad2, sigma=sigma2)
    grad2_transformed, grad_trend_line = linear_transform(grad2_denoised)

    ingress_left = np.argmin(grad2_transformed[ingress_mean-edge_margin2:ingress_mean]) + ingress_mean-edge_margin2
    ingress_right = np.argmax(grad2_transformed[ingress_mean:ingress_mean+edge_margin2]) + ingress_mean

    egress_left = np.argmax(grad2_transformed[egress_mean-edge_margin2:egress_mean]) + egress_mean-edge_margin2
    egress_right = np.argmin(grad2_transformed[egress_mean:egress_mean+edge_margin2]) + egress_mean 

    kwargs['ingress_left'] = ingress_left
    kwargs['ingress_right'] = ingress_right
    kwargs['egress_left'] = egress_left
    kwargs['egress_right'] = egress_right

    return kwargs


def split_by_phases(**kwargs):
    margin = kwargs['margin']
    signal = kwargs['signal']
    ingress_left = kwargs['ingress_left']
    ingress_right = kwargs['ingress_right']
    egress_left = kwargs['egress_left']
    egress_right = kwargs['egress_right']

    signal_before = signal[:ingress_left-margin]
    signal_transit = signal[ingress_right+margin:egress_left-margin]
    signal_after = signal[egress_right+margin:]

    signals = [signal_before, signal_transit, signal_after]
    kwargs['signals'] = signals

    return kwargs


# Wavelet denoising
def denoise_signal_with_wavelet(wavelet='db29', strength=1.0, mode='symmetric', level=None, **kwargs):
    signal = kwargs['signal']
    denoised_signal = wavelet_denoising(signal, wavelet=wavelet, strength=strength, mode=mode, level=level)
    kwargs['signal'] = denoised_signal
    return kwargs


def filter_signal_with_gaussian(sigma=10, **kwargs):
    signal = kwargs['signal']
    filtered_signal = gaussian_filter1d(signal, sigma=sigma)
    kwargs['signal'] = filtered_signal
    kwargs['signal_before_gaussian'] = signal
    return kwargs

def filter_signal_with_polyfit(extend_points=100, max_deg=4, **kwargs):
    signal = kwargs['signal']
    filtered_signal = polyfit_denoising(signal, extend_points=extend_points, max_deg=max_deg)
    kwargs['signal'] = filtered_signal
    return kwargs

def filter_signal_with_median(window=5, **kwargs):
    signal = kwargs['signal']
    filtered_signal = median_filter(signal, size=window)
    kwargs['signal'] = filtered_signal
    return kwargs

def denoise_signals_with_wavelet(wavelet='db29', strength=1.0, mode='symmetric', level=None, **kwargs):
    signals = kwargs['signals']
    denoised_signals = []
    for signal in signals:
        denoised_signal = wavelet_denoising(signal, wavelet=wavelet, strength=strength, mode=mode, level=level)
        denoised_signals.append(denoised_signal)
    kwargs['signals'] = denoised_signals
    return kwargs

def filter_signals_with_gaussian(sigma=10, **kwargs):
    signals = kwargs['signals']
    filtered_signals = []
    for signal in signals:
        filtered_signal = gaussian_filter1d(signal, sigma=sigma)
        filtered_signals.append(filtered_signal)
    kwargs['signals'] = filtered_signals
    return kwargs

def filter_signals_with_polyfit(extend_points=100, max_deg=4, **kwargs):
    signals = kwargs['signals']
    filtered_signals = []
    for signal in signals:
        filtered_signal = polyfit_denoising(signal, extend_points=extend_points, max_deg=max_deg)
        filtered_signals.append(filtered_signal)
    kwargs['signals'] = filtered_signals
    return kwargs

def reconstruct_signals(**kwargs):
    margin = kwargs['margin']
    signal_before, signal_transit, signal_after = kwargs['signals']
    ingress_left = kwargs['ingress_left']
    ingress_right = kwargs['ingress_right']
    egress_left = kwargs['egress_left']
    egress_right = kwargs['egress_right']

    signal_reconstructed = np.concatenate([
        signal_before, 
        np.ones(ingress_right-ingress_left + 2 * margin), 
        signal_transit, 
        np.ones(egress_right-egress_left + 2 * margin), 
        signal_after]
    )

    mask = np.ones_like(signal_reconstructed, dtype=bool)
    mask[ingress_left-margin:ingress_right+margin] = False
    mask[egress_left-margin:egress_right+margin] = False
    x = np.arange(signal_reconstructed.shape[0])
    x_reconstructed = x[mask]
    y_reconstructed = signal_reconstructed[mask]

    kwargs['x'] = x_reconstructed
    kwargs['signal'] = y_reconstructed
    return kwargs


def objective(adjustment, x, signal, ingress, egress, deg=4):
    s = adjustment[0]
    adjusted_signal = signal.copy()

    adjusted_signal[ingress:egress] *= (1 + s)

    coefficients = np.polyfit(x, adjusted_signal, deg=deg)
    polyfit_values = np.polyval(coefficients, x)

    mse = np.mean((adjusted_signal - polyfit_values) ** 2) * 1e3
    return mse

def fit_transit(deg=4, **kwargs):
    initial_guess = [0.0001]
    bounds = [(0, 1)]

    x = kwargs['x']
    signal = 100 * kwargs['signal']
    signals = kwargs['signals']
    ingress = len(signals[0])
    egress = len(signals[0]) + len(signals[1])
    kwargs['ingress'] = ingress
    kwargs['egress'] = egress
    result = minimize(lambda adjustment: objective(adjustment, x, signal, ingress, egress, deg=deg), 
                          initial_guess, bounds=bounds, method='Nelder-Mead')
    adjustment = result.x[-1]
    kwargs['adjustment'] = adjustment
    return kwargs


def get_features(**kwargs):
    features = []
    for feature in kwargs['features']:
        feature = kwargs[feature]
        if type(feature) == list:
            features.extend(feature)
        else:
            features.append(feature)
    kwargs['features'] = features
    return kwargs

def fit_transit_poly(**kwargs):
    sig = kwargs['signal'][kwargs['ingress_right']+0:kwargs['egress_left']-0] + 1
    x_fitted = np.arange(len(sig))
    extend_points = 200
    x_fitted, sig2 = extend_signal_with_polyfit(x_fitted, sig, extend_points, deg=2)
    x_fitted = np.arange(len(sig2)) 
    coefficients = np.polyfit(x_fitted, sig2, 3)
    fitted_curve = np.polyval(coefficients, x_fitted)
    fitted_curve = fitted_curve[extend_points:-extend_points]
    x_fitted = np.arange(len(fitted_curve)) + result['ingress_right']
    
    kwargs['transit_poly_coeffs'] = coefficients
    kwargs['transit_poly_signal'] = fitted_curve
    return kwargs

def collect_features(**kwargs):
    transit_start = kwargs['ingress_right']
    transit_finish = kwargs['egress_left']
    signal = kwargs['signal'] + 1
    signal_transit = signal[transit_start:transit_finish]
    transit_poly_signal = kwargs['transit_poly_signal']
    all_s = [] 
    all_s.append(signal_transit.mean())
    all_s.append(kwargs['transit_poly_coeffs'][-1])
    all_s.append(transit_poly_signal.mean())
    all_s.append(transit_poly_signal[len(transit_poly_signal) // 2])
    all_s.append(signal_transit.min())
    all_s.append(signal_transit.max())
    percentiles = np.percentile(signal_transit, np.arange(35, 65, 5))
    all_s.extend(percentiles)    
    
    features = []
    for s in all_s:
        x = transit_formula(s)
        features.append(x)

    features.append(signal_transit.std())
    features.extend(kwargs['transit_poly_coeffs'].tolist())
    
    ingress_signal = kwargs['signal'][kwargs['ingress_left']:kwargs['ingress_right']] + 1
    egress_signal = kwargs['signal'][kwargs['egress_left']:kwargs['egress_right']] + 1
    features.extend([ingress_signal.max(), ingress_signal.min(), egress_signal.max(), egress_signal.min()])
    features.append(signal_transit.mean())
    features.append(kwargs['transit_poly_coeffs'][-1])
    features.append(transit_poly_signal.mean())
    features.append(transit_poly_signal[len(transit_poly_signal) // 2])
    
    kwargs['features'] = features
    
    return kwargs

def collect_wavebin_features(**kwargs):
    transit_start = kwargs['ingress_right']
    transit_finish = kwargs['egress_left']
    signal = kwargs['signal'] + 1
    signal_transit = signal[transit_start:transit_finish]
    transit_poly_signal = kwargs['transit_poly_signal']
    all_s = [] 
    all_s.append(signal_transit.mean())
    all_s.append(kwargs['transit_poly_coeffs'][-1])
    all_s.append(transit_poly_signal.mean())
    all_s.append(transit_poly_signal[len(transit_poly_signal) // 2])
    features = []
    for s in all_s:
        x = transit_formula(s)
        features.append(x)

    kwargs['features'] = features
    
    return kwargs


def collect_less_features(**kwargs):
    transit_start = kwargs['ingress_right']
    transit_finish = kwargs['egress_left']
    signal = kwargs['signal'] + 1
    signal_transit = signal[transit_start:transit_finish]
    transit_poly_signal = kwargs['transit_poly_signal']
    all_s = [] 
    all_s.append(signal_transit.mean())
    all_s.append(kwargs['transit_poly_coeffs'][-1])
    all_s.append(transit_poly_signal.mean())
    all_s.append(transit_poly_signal[len(transit_poly_signal) // 2])
    all_s.append(signal_transit.min())
    all_s.append(signal_transit.max())
    percentiles = np.percentile(signal_transit, np.arange(35, 65, 5))
    all_s.extend(percentiles)    
    
    features = []
    for s in all_s:
        x = transit_formula(s)
        features.append(x)

    kwargs['features'] = features
    
    return kwargs

def collect_less_wavebin_features(**kwargs):
    transit_start = kwargs['ingress_right']
    transit_finish = kwargs['egress_left']
    signal = kwargs['signal'] + 1
    signal_transit = signal[transit_start:transit_finish]
#     transit_poly_signal = kwargs['transit_poly_signal']
    all_s = [] 
    all_s.append(signal_transit.mean())
#     all_s.append(kwargs['transit_poly_coeffs'][-1])
#     all_s.append(transit_poly_signal.mean())
#     all_s.append(transit_poly_signal[len(transit_poly_signal) // 2])
    features = []
    for s in all_s:
        x = transit_formula(s)
        features.append(x)

    kwargs['features'] = features
    
    return kwargs


In [6]:
basic_pipeline_phases = [
    lambda  **kwargs: bin_wavelengths(**kwargs),
    lambda  **kwargs: normalize_signal(**kwargs),
    lambda **kwargs: filter_signal_with_gaussian(sigma=25, **kwargs),
    lambda  **kwargs: get_transit_phases2(**kwargs),
#     lambda **kwargs: polynomial_detrend(**kwargs),
#     lambda **kwargs: fit_transit_poly(**kwargs),
#     lambda **kwargs: collect_features(**kwargs),
]

full_pipeline = [
    lambda  **kwargs: bin_wavelengths(**kwargs),
    lambda  **kwargs: normalize_signal(**kwargs),
    lambda **kwargs: filter_signal_with_gaussian(sigma=25, **kwargs),
    lambda **kwargs: polynomial_detrend(**kwargs),
    lambda **kwargs: fit_transit_poly(**kwargs),
    lambda **kwargs: collect_less_features(**kwargs),
]

wavebins_pipeline = [
    lambda  **kwargs: bin_wavelengths(**kwargs),
    lambda  **kwargs: normalize_signal(**kwargs),
    lambda **kwargs: filter_signal_with_gaussian(sigma=25, **kwargs),
    lambda **kwargs: polynomial_detrend(**kwargs),
#     lambda **kwargs: fit_transit_poly(**kwargs),
    lambda **kwargs: collect_less_wavebin_features(**kwargs),
]


In [7]:

initial_kwargs_39 = {'signal': None, 'margin': 1, 'from_wavelength':39, 'to_wavelength':321}
initial_kwargs_55 = {'signal': None, 'margin': 1, 'from_wavelength':55, 'to_wavelength':300}

pipelines = []
pipelines.append((full_pipeline, initial_kwargs_39))
pipelines.append((full_pipeline, initial_kwargs_55))

binning = 6
for i in range(282):
    from_wavelength = 39+i-binning//2
    to_wavelength = 39+i+binning//2
    initial_kwargs = initial_kwargs_39.copy()
    initial_kwargs['from_wavelength'] = from_wavelength
    initial_kwargs['to_wavelength'] = to_wavelength

    pipeline = (wavebins_pipeline, initial_kwargs)
    pipelines.append(pipeline)
        
print(len(pipelines))

284


In [8]:
n_planets = len(adc_info)
n_wavelengths = 282
all_s = []
all_features = []
feature_dicts = []

# n_planets = 2

for planet_id in tqdm(range(n_planets)):
    signal = preproc_airs(int(planet_ids[planet_id]), adc_info, dataset=mode)
    initial_kwargs = initial_kwargs_39.copy()
    initial_kwargs['signal'] = signal
    result = reduce(lambda kw, func: func(**kw), basic_pipeline_phases, initial_kwargs)
    ingress_left, ingress_right, egress_left, egress_right = result['ingress_left'], result['ingress_right'], result['egress_left'], result['egress_right']

    planet_features = []
    for pipeline in pipelines:
        initial_kwargs = pipeline[1].copy()
        initial_kwargs['signal'] = signal
        initial_kwargs['ingress_left'], initial_kwargs['ingress_right'], initial_kwargs['egress_left'], initial_kwargs['egress_right'] = ingress_left, ingress_right, egress_left, egress_right
        result = reduce(lambda kw, func: func(**kw), pipeline[0], initial_kwargs)

        planet_features.extend(result['features'])

    all_features.append(planet_features)
np.array(all_features).shape

100%|██████████| 1/1 [00:08<00:00,  8.43s/it]


(1, 306)

In [9]:
X = np.array(all_features)
scaler = joblib.load('/kaggle/input/ad24-head-10/models/scaler.pkl')
X_scaled = scaler.transform(X)
X.shape

(1, 306)

In [10]:
def predict_with_models(k, X, directory='/kaggle/input/ad24-head-10/models/'):
    all_predictions = []
    
    for i in tqdm(range(k)):
        model = None
        with open(f'{directory}model_{i}.pkl', 'rb') as file:
            model = joblib.load(file)
        preds = model.predict(X)
        all_predictions.append(preds)
        del model
        gc.collect()

    all_predictions = np.array(all_predictions)

    final_predictions = []
    std_devs = []

    for i in range(all_predictions.shape[1]): 
        sample_preds = all_predictions[:, i, :]
        sorted_preds = np.sort(sample_preds, axis=0)

        MARGINS = int(k * 0.45)
        filtered_preds = sorted_preds[MARGINS:-MARGINS] if sorted_preds.shape[0] > MARGINS*2+3 else sorted_preds
        
        mean_preds = np.mean(filtered_preds, axis=0)
        std_preds = np.std(all_predictions[:, i, :], axis=0)

        final_predictions.append(mean_preds)
        std_devs.append(std_preds)

    return np.array(final_predictions), np.array(std_devs)

transit_depths, sigma = predict_with_models(min(1000, 7000), X_scaled)

100%|██████████| 1000/1000 [02:51<00:00,  5.84it/s]


In [11]:
sample_submission = pd.read_csv('/kaggle/input/ariel-data-challenge-2024/sample_submission.csv')
pred = transit_depths.clip(0) 

# linear_distribution = np.linspace(0.00016, 0.00013, transit_depths.shape[1])
# sigma_response = np.tile(linear_distribution, (transit_depths.shape[0], 1))
# percentage_distribution = np.linspace(0.022, 0.022, pred.shape[1]) # Recommended by calculations
# sigma = percentage_distribution * pred

COEFF_0 = 2.9 # as predicted by the model
sigma_response = sigma * COEFF_0

submission = pd.DataFrame(np.concatenate([pred, sigma_response.clip(0)], axis=1), columns=sample_submission.columns[1:])
submission.index = adc_info.index
# submission.to_csv('submission.csv')
submission.to_csv('submission.csv', float_format='%.8f')

In [12]:
# !cat submission.csv